# Using PySpark with TimeCopilot

When using PySpark Java needs to be installed with a `JAVA_HOME` environment variable set. The environment variable may have been set when installing Java.

The required Java version may vary depending on the PySpark version in use.
For this example PySpark version `4.0.2` was used with OpenJDK version `21.0.10` used as the Java version.

In [2]:
import nest_asyncio

nest_asyncio.apply()

from timecopilot import TimeCopilotForecaster

from pyspark.sql import SparkSession
import pandas as pd

from timecopilot.models import SeasonalNaive
from timecopilot.models.foundation.chronos import Chronos

 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.


checking the `JAVA_HOME` environment variable.

In [3]:
import os
print(os.environ['JAVA_HOME'])

/opt/homebrew/opt/openjdk@21/libexec/openjdk.jdk/Contents/Home


## Start the Spark session

In [4]:

spark = SparkSession.builder.getOrCreate()

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/19 18:55:32 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


## Create the dataframe

In [10]:
df = pd.read_csv("https://timecopilot.s3.amazonaws.com/public/data/air_passengers.csv", parse_dates=["ds"])
s_df = spark.createDataFrame(df)
print(s_df)
s_df.show(n=5)

DataFrame[unique_id: string, ds: timestamp, y: bigint]
+-------------+-------------------+---+
|    unique_id|                 ds|  y|
+-------------+-------------------+---+
|AirPassengers|1949-01-01 00:00:00|112|
|AirPassengers|1949-02-01 00:00:00|118|
|AirPassengers|1949-03-01 00:00:00|132|
|AirPassengers|1949-04-01 00:00:00|129|
|AirPassengers|1949-05-01 00:00:00|121|
+-------------+-------------------+---+
only showing top 5 rows


## Create a TimeCopilotForecaster

In [11]:
tcf = TimeCopilotForecaster(
    models=[
        SeasonalNaive(),
        Chronos(repo_id="autogluon/chronos-2-small")
    ]
)

## Create a forecast

In [12]:
result = tcf.forecast(
    df=s_df,
    h=12
)

2026-03-18 17:02:20,681	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.
2026-03-18 17:02:20,745	INFO util.py:154 -- Missing packages: ['ipywidgets']. Run `pip install -U ipywidgets`, then restart the notebook server for rich notebook output.


## View forecast results

In [13]:
print(result)
result.show(n=12)

DataFrame[unique_id: string, ds: timestamp, SeasonalNaive: double, Chronos: double]


 See https://github.com/google-research/timesfm/blob/master/README.md for updated APIs.
  0%|          | 0/1 [00:00<?, ?it/s]

+-------------+-------------------+-------------+-------+
|    unique_id|                 ds|SeasonalNaive|Chronos|
+-------------+-------------------+-------------+-------+
|AirPassengers|1961-01-01 00:00:00|        417.0|  454.0|
|AirPassengers|1961-02-01 00:00:00|        391.0|  430.0|
|AirPassengers|1961-03-01 00:00:00|        419.0|  506.0|
|AirPassengers|1961-04-01 00:00:00|        461.0|  504.0|
|AirPassengers|1961-05-01 00:00:00|        472.0|  512.0|
|AirPassengers|1961-06-01 00:00:00|        535.0|  584.0|
|AirPassengers|1961-07-01 00:00:00|        622.0|  660.0|
|AirPassengers|1961-08-01 00:00:00|        606.0|  668.0|
|AirPassengers|1961-09-01 00:00:00|        508.0|  568.0|
|AirPassengers|1961-10-01 00:00:00|        461.0|  488.0|
|AirPassengers|1961-11-01 00:00:00|        390.0|  418.0|
|AirPassengers|1961-12-01 00:00:00|        432.0|  468.0|
+-------------+-------------------+-------------+-------+



100%|██████████| 1/1 [00:01<00:00,  1.65s/it]


/Users/shane/.local/share/uv/python/cpython-3.11.14-macos-aarch64-none/lib/python3.11/multiprocessing/resource_tracker.py:254: UserWarning: resource_tracker: There appear to be 1 leaked semaphore objects to clean up at shutdown
  warnings.warn('resource_tracker: There appear to be %d '
